# Réorganisation du projet et séparation temporelle
Ce carnet montre comment restructurer le pipeline à partir du *notebook* initial
et comment effectuer une séparation train/test **par date** adaptée aux séries
temporelles.

## 1. Importer les bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# modules réutilisables du package src
from src.data.load_dataset import load_dataset
from src.data.aggregation import aggregate_daily
from src.data.cleaning import clean_dataset
from src.features.clearsky_features import add_clearsky_to_df_model
from src.features.kc_features import add_kc_to_df
from src.features.time_features import add_time_features
from src.models.split import train_test_split_by_date
from src.models.predict import scale_features, make_predictions
from src.models.train_linear_model import train_model
from src.models.evaluate_model import evaluate_model
from src.visualization.plot_predictions import plot_one_test_day_bar_plots, plot_real_vs_pred

## 2. Charger et afficher les données

In [ ]:
df = load_dataset('data/raw/df_model_Pierre_ozoir.pkl')
df.head()

## 3. Prétraitement : gérer les dates et trier
La colonne d'index est déjà un `DatetimeIndex`. On ajoute clearsky et kc,
puis on agrège quotidiennement et on nettoie.

In [ ]:
# caractéristiques horaires -> journalières

df = add_clearsky_to_df_model(df)
df = add_kc_to_df(df, ghi_col='shortwave_radiation')
df_daily = aggregate_daily(df)

df_clean = clean_dataset(df_daily, min_production_wh=1000.0)
df_final = add_time_features(df_clean)

df_final.head()

## 4. Séparation temporelle des ensembles
On choisit une date de coupure plutôt qu'un échantillonnage aléatoire.

In [ ]:
target = 'PV_ELE_IME_I_001'
features = ['direct_radiation','diffuse_radiation','shortwave_radiation',
            'cs_ghi','kc_day','day_of_year_sin','day_of_year_cos']

X = df_final[features]
y = df_final[target]

split_date = '2020-07-31'
X_train, X_test, y_train, y_test = train_test_split_by_date(X, y, split_date)

print(f"train {len(X_train)} samples, test {len(X_test)} samples")

## 5. Construction du modèle de régression linéaire

In [ ]:
scaler, X_train_s, X_test_s = scale_features(X_train, X_test)
model = train_model(X_train_s, y_train)  # linear regression par défaut

## 6. Évaluation du modèle

In [ ]:
y_pred = make_predictions(model, X_test_s)
r2, mae, rmse = evaluate_model(y_test, y_pred)

print(f"R²={r2:.3f}, MAE={mae:.1f}, RMSE={rmse:.1f}")

plot_one_test_day_bar_plots(str(X_test.index[0].date()), y_test, y_pred)
plot_real_vs_pred(y_test, y_pred)